In [8]:
import csv, time, json, ast
from seleniumbase import Driver
from pprint import pprint
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import shutil
import csv
import itertools

In [2]:
extension_dir = fr"C:\Users\admin\AppData\Local\Google\Chrome\User Data\Profile 9\Extensions\majdfhpaihoncoakbjgbdhglocklcgno\3.0.0_0" ## Muaz's PC
extension_sub_dir = next(os.walk(extension_dir))[1][0]
extension_dir += extension_sub_dir

browser = Driver(uc=True, extension_dir=extension_dir)
time.sleep(5)

In [9]:
brand = 'tre'
browser = Driver(uc=True, incognito=True)
wait = WebDriverWait(browser, 10)
# browser.maximize_window()

In [10]:
browser.get(f'https://autoid.co/')

In [7]:
header = ['Link']
products_list = []

with open(f'TRE-products.csv', 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(header)

    try:
        proceed = True
        current_page = 1

        while(proceed):
            url_product_page = "https://autoid.co/collections/tre?page=" + str(current_page)
            browser.get(url_product_page)
            time.sleep(2)

            products = browser.find_elements(by=By.CSS_SELECTOR, value='.product-card__title a')
            for i in products:
                product = i.get_attribute('href')
                writer.writerow([product])
                products_list.append(product)
            products_list = list(set(products_list))
            print(len(products_list))

            product_existence = browser.find_elements(by=By.CSS_SELECTOR, value='.product-card__title a')
            if product_existence == []:
                proceed = False
            else:
                current_page += 1
        
    except Exception as e:
        print(f'{str(e).splitlines()[0]}')

24
48
72
96
120
144
168
176
176


In [11]:
missed_sku = []
header = ['link', 'title', 'sku', 'cleaned_data_list', 'clean_desc', 'tabcontent', 'image_list']
with open(f'TRE-scrape.csv', 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(header)

    df1 = pd.read_csv(r"TRE-products.csv", dtype=str)
    for index, row in df1.iterrows():
        link = row['Link']
        
        browser.get(f'{link}')
        time.sleep(2)
        try:
            title = browser.find_element(by=By.CSS_SELECTOR, value='h1.product-info__title').get_attribute('innerText').strip()

            try:
                sku = browser.find_element(by=By.CSS_SELECTOR, value='.product-info__sku').get_attribute('innerText').strip()
            except:
                sku = ''

            try:
                script_list = []
                scripts = browser.find_elements(by=By.CSS_SELECTOR, value='script')
                for i in scripts:
                    script = i.get_attribute('innerHTML').strip()
                    script_list.append(script)

                variants = [item for item in script_list if 'window.ShopifyAnalytics.meta[attr]' in item]
                variant = variants[0]
                # print(variant)

                var = json.loads(variant.split('"variants":')[-1].split(',"remote":')[0].strip())
                # print(var)

                cleaned_data_list = [
                    {
                        "public_title": item["public_title"],
                        "price": item["price"],
                        "sku": item["sku"],
                    }
                    for item in var
                    ]

                # print(cleaned_data_list)
            except:
                cleaned_data_list = []

            try:
                desc = browser.find_element(by=By.CSS_SELECTOR, value='.station-tabs-local-above').get_attribute('innerHTML').strip()
                clean_desc = re.sub(r'<(\w+)(\s+[^>]*?)?>', r'<\1>', desc)
            except:
                clean_desc = ''

            try:
                tab_title_list = []
                tab_titles = browser.find_elements(by=By.CSS_SELECTOR, value='h4.station-tabs-tabtitle .station-tabs-tabtext')
                for tab in tab_titles:
                    tab_title = tab.get_attribute('innerText').replace("\n", "").strip()
                    tab_title_list.append(tab_title)
                
                tab_info_list = []
                tab_infos = browser.find_elements(by=By.CSS_SELECTOR, value='.station-tabs-tabpanel .station-tabs-tabcontent')
                for tab in tab_infos:
                    tab_info = tab.get_attribute('innerHTML').replace("\n", "").strip()
                    tab_info = re.sub(r'<(\w+)(\s+[^>]*?)?>', r'<\1>', tab_info)

                    tab_info_list.append(tab_info)

                tabcontent = dict(zip(tab_title_list, tab_info_list))                

            except:
                tabcontent = {} 

            images = browser.find_elements(by=By.CSS_SELECTOR, value='.product-gallery__media img')
            image_list = []
            for i in images:
                image = i.get_attribute('src')
                image_list.append(image)

            writer.writerow([link, title, sku, cleaned_data_list, clean_desc, tabcontent, image_list])
            print([link, title, sku, cleaned_data_list, clean_desc, tabcontent, image_list])

        except Exception as e:
            print(f"{link}: {e}")
            missed_sku.append(link)

print(missed_sku)

['https://autoid.co/products/tre-limited-edition-faraday-box', 'TRE KEY FARADAY BOX', 'SKU: TR-FB', [{'public_title': None, 'price': 2465, 'sku': 'TR-FB'}], '<p><strong>TRE Limited Edition Wireless Car Key Protection Faraday Box</strong></p>\n<p>Introducing the <strong>TRE Limited Edition Wireless Car Key Protection Faraday Box</strong>—where cutting-edge security meets refined sophistication. Crafted for discerning drivers, this isn’t just a protective accessory—it’s a statement of style and peace of mind. Wrapped in sumptuous <strong>Alcantara</strong>, with the iconic <strong>TRE logo</strong> elegantly embossed on top, this Faraday Box transforms ordinary key storage into a premium experience.</p>\n<p>Engineered to <strong>block thieves from stealing your car via signal extenders</strong>, it silently stands guard, giving you confidence while you go about your day. Its sleek, minimalist design fits seamlessly into your home, office, or vehicle interior, making security look effortl

In [12]:
csv_path = r"TRE-scrape.csv"
scrape_df = pd.read_csv(csv_path).fillna('')

final_data = []

for index0, row in scrape_df.iterrows():

    vendor = 'TRE'
    
    sku = row['sku']

    # title = f'{vendor} ' + sku + ' ' + row['title'].replace(' BY AP ', '')
    title = row['title']

    proper = f'{vendor} ' + sku + ' ' + row['title'].replace(' BY AP ', '').title()

    desc = row['clean_desc']
    if desc == '':
        desc = f'<p>This is {title}</p>'

    images = ast.literal_eval(row['image_list'])
    
    handle = (re.sub(r'[^a-zA-Z0-9\n\.]', '-', title).replace(".", "-").replace("---", "-").replace("--", "-")).lower()
    if handle.endswith('-'):
        handle = handle[:-1]

    variations = ast.literal_eval(row['cleaned_data_list'])

    tabcontent = ast.literal_eval(row['tabcontent'])

    try:
        what_inc = "<p>&nbsp;</p><h4>What's Included</h4>" + tabcontent["WHAT'S IN THE BOX"]
    except:
        what_inc = ''

    try:
        features = "<p>&nbsp;</p><h4>Features</h4>" + tabcontent["KEY FEATURES:"]
    except:
        features = ''

    part_no = ''
    for variant in variations:
        part_no += f"<p>AUTOMOTIVE-PASSION-{variant['sku']} ({variant['public_title']})</p>"

    #######images & variations comparison#######
    len_image = len(images)
    len_variant = len(variations)

    diff = abs(len_image - len_variant)

    if len_image > len_variant:
        for _ in range(diff):
            variations.append({key: '' for key in variations[0].keys()})
    elif len_variant > len_image:
        images.extend(['']*diff)
    

    def description(desc, part_no, vendor, what_inc, features):
        return f"""<h4><strong>Description</strong></h4>{desc}
        {features}
        {what_inc}
        <p>&nbsp;</p>
        <h4>Compatibility</h4><p>Feel free to contact us at info@mlperformance.co.uk should you wish to double check!</p>
        <p>&nbsp;</p>
        <h4>Compatibility Check</h4><p>To ensure the part(s) you have ordered fits your vehicle, we run a compatibility check prior to dispatch. We can do this either using your registration number (UK) or the last 7 digits of your VIN. Simply enter your car details prior to checkout.</p>
        <p>&nbsp;</p>
        <h4>Part Number</h4>{part_no}
        <p>&nbsp;</p>
        <h4>More Information</h4><p><strong>Manufactured by</strong></p><p>{vendor}</p>"""

    # Loop through images
    for index, image in enumerate(images, start=0):
        option_value1 = variations[index]['public_title']
        variant_sku = variations[index]['sku']
        variant_price = variations[index]['price']

        # Create a new dictionary for each image
        info = {}
        
        info['Handle'] = handle
        info['Title'] = title
        info['Body (HTML)'] = description(desc, part_no, vendor, what_inc, features).replace("\n", "")
        info['Vendor'] = f'{vendor}'
        info['Standardized Product Type'] = proper
        info['Custom Product Type'] = None
        info['Tags'] = f"Uploaded by_Muazzim, MLP Discount 2025, Brand_{vendor}, 2025_New Products, Product Type_"
        info['Published'] = "TRUE"
        info['Option1 Name'] = ''
        info['Option1 Value'] = option_value1
        info['Option2 Name'] = ''
        info['Option2 Value'] = ''
        info['Option3 Name'] = ''
        info['Option3 Value'] = ''

        info['Variant SKU'] = f'AUTOMOTIVE-PASSION-{variant_sku}'
        info['Variant Grams'] = ''
        info['Variant Inventory Tracker'] = "shopify"
        info['Variant Inventory Policy'] = 'continue'
        info['Variant Fulfillment Service'] = 'manual'
        info['Variant Price'] = None
        info['Variant Compare At Price'] = variant_price
        info['Variant Requires Shipping'] = 'TRUE'
        info['Variant Taxable'] = 'TRUE'
        info['Variant Barcode'] = variant_sku

        # For each image, create a new entry
        info['Image Src'] = image
        info['Image Position'] = index + 1
        info['Image Alt Text'] = title
        info['Gift Card'] = None
        info['SEO Title'] = title
        info['SEO Description'] = 'Get ' + title + ' for your car to get your desired looks and performance from ML Performance at the lowest price with FREE UK shipping & next day delivery on in stock items. Very cheap prices & good service.'
        info['Google Shopping / Google Product Category'] = None
        info['Google Shopping / Gender'] = None
        info['Google Shopping / Age Group'] = None
        info['Google Shopping / MPN'] = variant_sku
        info['Google Shopping / AdWords Grouping'] = None
        info['Google Shopping / AdWords Labels'] = None
        info['Google Shopping / Condition'] = 'new'
        info['Google Shopping / Custom Product'] = None
        info['Google Shopping / Custom Label 0'] = None
        info['Google Shopping / Custom Label 1'] = None
        info['Google Shopping / Custom Label 2'] = None
        info['Google Shopping / Custom Label 3'] = None
        info['Google Shopping / Custom Label 4'] = None
        info['Variant Image'] = None
        info['Variant Weight Unit'] = 'kg'
        info['Variant Tax Code'] = 8708949900
        info['Cost per item'] = None
        info['Margins'] = None
        info['Price / International'] = None
        info['Compare At Price / International'] = None
        info['Status'] = 'active'

        # Append the new dictionary to the final_data list
        final_data.append(info)

final_df = pd.DataFrame(final_data)

output_path = os.path.join("TRE-HTML.csv")
final_df.to_csv(output_path, index=False)

print('File saved and moved to desired folder')

File saved and moved to desired folder
